
# Bursty continuity prior: bin-edge-dependent σ schedule (Tacchella+2022)

The bursty continuity prior (Tacchella+2022, ApJ 926, 134) shares the
piecewise-constant continuity SFH with Leja+2019 but doubles the
Student-t scale on log-SFR ratios whose younger bin edge is *recent*
(< 1 Gyr lookback). The result is a prior that lets recent SFR variations
swing by ~1 dex while keeping older history smooth (σ = 0.3 dex).

This example does two things:

1. **Top panel** — prints (and plots) the σ schedule for the default
   7-bin grid ``DEFAULT_BIN_EDGES_GYR``. Ratios 0–2 sit in the bursty
   regime (σ=1.0); ratios 3–5 sit in the smooth regime (σ=0.3).
2. **Bottom panel** — draws 60 SFR trajectories each from
   ``continuity`` and ``bursty_continuity`` so the visual width of the
   recent-time bands directly shows the σ-doubling effect.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri import Parameters, SEDModel
from tengri.analysis.plotting import setup_style
from tengri.sfh import DEFAULT_BIN_EDGES_GYR

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# ``bursty_continuity`` is available for SFH sampling and the prior shape
# it draws is the whole point of this example. Build it via the flat-kwarg
# ``Parameters(mean_sfh_type=...)`` form and call ``predict_sfh`` to evaluate
# the star-formation history alone.
N_DRAWS = 60

# ── σ schedule ────────────────────────────────────────────────────
edges = np.asarray(DEFAULT_BIN_EDGES_GYR)
sigmas = []
spec = Parameters(mean_sfh_type="bursty_continuity", redshift=0.0)
for i in range(6):
    prior = spec.get_distribution(f"sfh_burstcont_ratio_{i}")
    # StudentT exposes no sigma attribute; it prints as
    # StudentT(mu=0.0, sigma=X, df=2.0).
    sigmas.append(float(repr(prior).split("sigma=")[1].split(",")[0]))
sigmas = np.asarray(sigmas)

# ── prior draws (via predict_sfh) ───────────────────────────────────────
ssp = tengri.load_ssp()
key0 = jax.random.PRNGKey(13)


def _model_for(sfh_type):
    """Build an SFH-only model, bypassing the high-level builder's type gate.

    ``Parameters(mean_sfh_type=...)`` accepts gated (DSPS-unvalidated) SFHs; the
    gate lives only in ``SEDModel.build``. ``precompute=False`` skips the SPS
    LUT we never touch — only ``predict_sfh`` is called.
    """
    spec = Parameters(mean_sfh_type=sfh_type, redshift=0.0)
    return SEDModel(spec=spec, ssp_data=ssp, precompute=False)


def _sample_normalized_sfh(model, n_draws, key):
    out = []
    for sub_key in jax.random.split(key, n_draws):
        p = dict(model.spec.sample(sub_key))
        sfh = model.predict_sfh(p)
        t = np.asarray(sfh["t_gyr"])
        sfr = np.asarray(sfh["sfr_mean"])
        mass = float(np.trapezoid(sfr, t * 1.0e9))
        if mass > 0:
            sfr = sfr / mass
        out.append(sfr)
    return t, np.asarray(out)


t, sfr_cont = _sample_normalized_sfh(_model_for("continuity"), N_DRAWS, key0)
_, sfr_burst = _sample_normalized_sfh(_model_for("bursty_continuity"), N_DRAWS, key0)

# ── figure ────────────────────────────────────────────────────────
fig, (ax_sig, ax_sfh) = plt.subplots(
    2, 1, figsize=(8.0, 7.0), gridspec_kw={"height_ratios": [1.0, 2.4], "hspace": 0.25}
)

# σ schedule
younger_edges_gyr = edges[1:-1]
ax_sig.step(younger_edges_gyr, sigmas, where="post", color="#cc4477", lw=2.0)
ax_sig.axvline(1.0, color="0.5", ls="--", lw=0.8)
ax_sig.text(1.05, 0.55, "t_split = 1 Gyr", color="0.4", fontsize=8)
ax_sig.set_xscale("log")
ax_sig.set_xlabel("younger edge of ratio (Gyr lookback)")
ax_sig.set_ylabel(r"StudentT scale  $\sigma$  [dex]")
ax_sig.set_ylim(0.0, 1.2)
ax_sig.set_title("Bursty σ schedule on the default 7-bin grid", fontsize=10)

# prior draws
for sfr in sfr_cont:
    ax_sfh.plot(t, sfr, color="#3388aa", lw=0.4, alpha=0.25)
for sfr in sfr_burst:
    ax_sfh.plot(t, sfr, color="#cc4477", lw=0.4, alpha=0.25)
ax_sfh.plot([], [], color="#3388aa", lw=1.4, label="continuity (Leja+19, σ=0.3)")
ax_sfh.plot([], [], color="#cc4477", lw=1.4, label="bursty (Tacchella+22, σ=1.0/0.3)")
ax_sfh.set_yscale("log")
ax_sfh.set_xlim(0, 13.5)
ax_sfh.set_ylim(1e-12, 5e-9)
ax_sfh.set_xlabel(r"lookback time  [Gyr]")
ax_sfh.set_ylabel(r"SFR$(t)$ / $M_\star^{\rm tot}$  [yr$^{-1}$]")
ax_sfh.legend(frameon=False, fontsize=9, loc="lower left")

plt.savefig("plot_bursty_continuity_sigma_schedule.png", dpi=150, bbox_inches="tight")